# Gestra — Reproduce All Figures\n\nThis notebook generates every figure used in the final report.\nAll outputs are saved to `results/figures/`.\n\nRequirements: run from the project root's `.venv` (see README).

In [1]:
import sys, pathlib, json, time
import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
import pandas as pd

PROJECT_ROOT = pathlib.Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

FIGURES_DIR = PROJECT_ROOT / "results" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 150, "savefig.dpi": 150, "savefig.bbox": "tight"})
print(f"Project root: {PROJECT_ROOT}")
print(f"Figures dir:  {FIGURES_DIR}")

Project root: /Users/ivanj/Desktop/Gesture game/Gestra_final
Figures dir:  /Users/ivanj/Desktop/Gesture game/Gestra_final/results/figures


## 1. Dataset Summary

In [2]:
from ml.train_personal import PersonalDataset, ACTION_NAMES, ACTION_MAP

data_root = PROJECT_ROOT / "data" / "personal"

# Count raw recordings and frames per class
rec_counts = {}
frame_counts = {}
for cls_name in ACTION_NAMES:
    cls_dir = data_root / cls_name
    if not cls_dir.exists():
        rec_counts[cls_name] = 0
        frame_counts[cls_name] = 0
        continue
    npz_files = sorted(cls_dir.glob("*.npz"))
    rec_counts[cls_name] = len(npz_files)
    total_frames = 0
    for f in npz_files:
        with np.load(f) as d:
            total_frames += d["valid"].astype(bool).sum()
    frame_counts[cls_name] = int(total_frames)

# Count sliding windows
ds = PersonalDataset(data_root, window=20, stride=3)
window_counts = {name: 0 for name in ACTION_NAMES}
for _, label in ds.entries:
    window_counts[ACTION_NAMES[label]] += 1

print("Recordings per class:", rec_counts)
print("Valid frames per class:", frame_counts)
print("Sliding windows per class:", window_counts)
print(f"Total windows: {len(ds)}")

# Plot
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3"]

axes[0].bar(ACTION_NAMES, [rec_counts[n] for n in ACTION_NAMES], color=colors)
axes[0].set_title("Recordings per Class")
axes[0].set_ylabel("Count")

axes[1].bar(ACTION_NAMES, [frame_counts[n] for n in ACTION_NAMES], color=colors)
axes[1].set_title("Valid Frames per Class")
axes[1].set_ylabel("Count")

axes[2].bar(ACTION_NAMES, [window_counts[n] for n in ACTION_NAMES], color=colors)
axes[2].set_title("Sliding Windows per Class (w=20, s=3)")
axes[2].set_ylabel("Count")

for ax in axes:
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "dataset_summary.png")
plt.show()
print(f"Saved: {FIGURES_DIR / 'dataset_summary.png'}")

Recordings per class: {'idle': 4, 'lpunch': 1, 'rpunch': 1, 'forward': 1, 'backward': 1}
Valid frames per class: {'idle': 610, 'lpunch': 278, 'rpunch': 278, 'forward': 231, 'backward': 231}
Sliding windows per class: {'idle': 1079, 'lpunch': 361, 'rpunch': 347, 'forward': 319, 'backward': 321}
Total windows: 2427


Saved: /Users/ivanj/Desktop/Gesture game/Gestra_final/results/figures/dataset_summary.png


/var/folders/_1/p0t70_y97x96fvjdnly6tc8h0000gn/T/ipykernel_2507/1429408990.py:54: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Training Curve (Baseline LR-A)

In [3]:
epoch_log_path = PROJECT_ROOT / "motion" / "models" / "epochs_LR-A.json"
with open(epoch_log_path) as f:
    epoch_log = json.load(f)

epochs = [e["epoch"] for e in epoch_log]
train_acc = [e["train_acc"] for e in epoch_log]
val_acc = [e["val_acc"] for e in epoch_log]
train_loss = [e["train_loss"] for e in epoch_log]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, train_acc, label="Train Accuracy", linewidth=1.5)
ax1.plot(epochs, val_acc, label="Val Accuracy", linewidth=1.5)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.set_title("Training & Validation Accuracy (LR-A Baseline)")
ax1.legend()
ax1.set_ylim(0.5, 1.05)
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, train_loss, label="Train Loss", linewidth=1.5, color="#C44E52")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.set_title("Training Loss")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "training_curve.png")
plt.show()
print(f"Saved: {FIGURES_DIR / 'training_curve.png'}")

Saved: /Users/ivanj/Desktop/Gesture game/Gestra_final/results/figures/training_curve.png


/var/folders/_1/p0t70_y97x96fvjdnly6tc8h0000gn/T/ipykernel_2507/2614719188.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Confusion Matrix

In [4]:
from ml.model import ActionTCN
from ml.train_personal import (normalize_seq, add_vel_acc, FEAT_DIM, NUM_CLASSES,
                                UPPER_BODY_JOINTS, NUM_JOINTS)

model_path = PROJECT_ROOT / "motion" / "models" / "action_personal.pt"
checkpoint = torch.load(model_path, map_location="cpu", weights_only=False)

model = ActionTCN(input_dim=FEAT_DIM, num_classes=checkpoint["num_classes"],
                  channels=(64, 64), kernel_size=5, dropout=0.0)
model.load_state_dict(checkpoint["state_dict"])
model.eval()

# Same train/val split as training (seed=42)
ds = PersonalDataset(data_root, window=20, stride=3)
n_val = max(1, int(len(ds) * 0.2))
n_train = len(ds) - n_val
_, val_ds = torch.utils.data.random_split(
    ds, [n_train, n_val], generator=torch.Generator().manual_seed(42))

y_true, y_pred = [], []
for idx in val_ds.indices:
    seq, label = ds.entries[idx]
    normed = normalize_seq(seq)
    selected = normed[:, UPPER_BODY_JOINTS, :]
    flat = selected.reshape(len(selected), NUM_JOINTS * 3)
    feat = add_vel_acc(flat)
    tensor = torch.from_numpy(feat).unsqueeze(0)
    with torch.no_grad():
        pred = model(tensor).argmax(1).item()
    y_true.append(label)
    y_pred.append(pred)

cm = confusion_matrix(y_true, y_pred)
print(classification_report(y_true, y_pred, target_names=ACTION_NAMES, zero_division=0))

fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(cm, display_labels=ACTION_NAMES)
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title("Confusion Matrix — Personal TCN Model (Val Split)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "confusion_matrix.png")
plt.show()
print(f"Saved: {FIGURES_DIR / 'confusion_matrix.png'}")

              precision    recall  f1-score   support

        idle       0.96      0.90      0.93       200
      lpunch       0.88      0.96      0.92        77
      rpunch       0.97      0.97      0.97        70
     forward       0.94      0.97      0.95        64
    backward       0.92      0.99      0.95        74

    accuracy                           0.94       485
   macro avg       0.94      0.96      0.95       485
weighted avg       0.94      0.94      0.94       485

Saved: /Users/ivanj/Desktop/Gesture game/Gestra_final/results/figures/confusion_matrix.png


/var/folders/_1/p0t70_y97x96fvjdnly6tc8h0000gn/T/ipykernel_2507/1124007338.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Rule-Based vs TCN Comparison

In [5]:
from ml.evaluate_offline import rule_based_predict_sequence

# Rule-based predictions on same val split
val_entries = [ds.entries[i] for i in val_ds.indices]

y_rule = []
for seq, label in val_entries:
    pred_name = rule_based_predict_sequence(seq)
    y_rule.append(ACTION_NAMES.index(pred_name))
y_rule = np.array(y_rule)
y_true_arr = np.array(y_true)
y_pred_arr = np.array(y_pred)

# Per-class accuracy
rule_per_class = []
tcn_per_class = []
for i, name in enumerate(ACTION_NAMES):
    mask = y_true_arr == i
    if mask.sum() == 0:
        rule_per_class.append(0)
        tcn_per_class.append(0)
    else:
        rule_per_class.append((y_rule[mask] == i).mean())
        tcn_per_class.append((y_pred_arr[mask] == i).mean())

rule_overall = (y_rule == y_true_arr).mean()
tcn_overall = (y_pred_arr == y_true_arr).mean()

# Plot
labels = ACTION_NAMES + ["Overall"]
rule_vals = rule_per_class + [rule_overall]
tcn_vals = tcn_per_class + [tcn_overall]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, rule_vals, width, label="Rule-Based", color="#C44E52", alpha=0.85)
bars2 = ax.bar(x + width/2, tcn_vals, width, label="TCN (Personal)", color="#4C72B0", alpha=0.85)

ax.set_ylabel("Accuracy")
ax.set_title("Rule-Based vs TCN — Per-Class and Overall Accuracy")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30)
ax.set_ylim(0, 1.15)
ax.legend()
ax.grid(axis="y", alpha=0.3)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{bar.get_height():.0%}", ha="center", va="bottom", fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{bar.get_height():.0%}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "rule_vs_tcn.png")
plt.show()
print(f"Rule-based overall: {rule_overall:.1%}")
print(f"TCN overall:        {tcn_overall:.1%}")
print(f"Saved: {FIGURES_DIR / 'rule_vs_tcn.png'}")

Rule-based overall: 45.6%
TCN overall:        94.0%
Saved: /Users/ivanj/Desktop/Gesture game/Gestra_final/results/figures/rule_vs_tcn.png


/var/folders/_1/p0t70_y97x96fvjdnly6tc8h0000gn/T/ipykernel_2507/3312128704.py:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Hyperparameter Experiment Results

In [6]:
csv_path = PROJECT_ROOT / "results" / "experiment_log.csv"
df = pd.read_csv(csv_path)
print(df.to_string(index=False))

# Filter to training runs only (exclude RULE and TCN-EVAL)
train_df = df[df["model"] == "ActionTCN"].copy()
train_df = train_df[train_df["run_id"].str.startswith(("LR-", "OPT-", "BS-", "WIN-"))].copy()

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# LR sweep
lr_df = train_df[train_df["run_id"].str.startswith("LR-")]
axes[0, 0].bar(lr_df["run_id"], lr_df["validation_accuracy"].astype(float), color="#4C72B0")
axes[0, 0].set_title("Learning Rate Sweep")
axes[0, 0].set_ylabel("Val Accuracy")
axes[0, 0].set_ylim(0.9, 1.02)
for i, row in lr_df.iterrows():
    axes[0, 0].text(list(lr_df["run_id"]).index(row["run_id"]), float(row["validation_accuracy"]) + 0.003,
                     f"lr={row['learning_rate']}", ha="center", fontsize=8)

# Optimizer comparison
opt_df = train_df[train_df["run_id"].str.startswith("OPT-") | (train_df["run_id"] == "LR-A")]
axes[0, 1].bar(["LR-A\n(Adam)", "OPT-B\n(AdamW)", "OPT-C\n(AdamW)"],
               opt_df["validation_accuracy"].astype(float).values, color="#DD8452")
axes[0, 1].set_title("Adam vs AdamW")
axes[0, 1].set_ylabel("Val Accuracy")
axes[0, 1].set_ylim(0.9, 1.02)

# Batch size
bs_df = train_df[train_df["run_id"].str.startswith("BS-") | (train_df["run_id"] == "LR-A")]
bs_labels = []
bs_vals = []
for rid in ["BS-A", "LR-A", "BS-C"]:
    row = train_df[train_df["run_id"] == rid].iloc[0]
    bs_labels.append(f"bs={int(row['batch_size'])}")
    bs_vals.append(float(row["validation_accuracy"]))
axes[1, 0].bar(bs_labels, bs_vals, color="#55A868")
axes[1, 0].set_title("Batch Size")
axes[1, 0].set_ylabel("Val Accuracy")
axes[1, 0].set_ylim(0.9, 1.02)

# Window length
win_labels = []
win_vals = []
for rid in ["WIN-A", "LR-A", "WIN-C"]:
    row = train_df[train_df["run_id"] == rid].iloc[0]
    win_labels.append(f"w={int(row['window_size'])}")
    win_vals.append(float(row["validation_accuracy"]))
axes[1, 1].bar(win_labels, win_vals, color="#8172B3")
axes[1, 1].set_title("Window Length")
axes[1, 1].set_ylabel("Val Accuracy")
axes[1, 1].set_ylim(0.9, 1.02)

for ax in axes.flat:
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("Hyperparameter Experiments — Validation Accuracy", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "hyperparameter_sweep.png")
plt.show()
print(f"Saved: {FIGURES_DIR / 'hyperparameter_sweep.png'}")

  run_id     model optimizer  learning_rate  batch_size  weight_decay  window_size  epochs  train_accuracy  validation_accuracy  training_time_s                     notes
    LR-A ActionTCN      adam         0.0010        16.0        0.0000           20    60.0          0.9949               1.0000             27.1                       NaN
    LR-B ActionTCN      adam         0.0003        16.0        0.0000           20    60.0          0.9899               1.0000             27.1                       NaN
    LR-C ActionTCN      adam         0.0001        16.0        0.0000           20    60.0          0.9924               1.0000             27.1                       NaN
   OPT-B ActionTCN     adamw         0.0010        16.0        0.0001           20    60.0          0.9975               1.0000             27.1                       NaN
   OPT-C ActionTCN     adamw         0.0003        16.0        0.0001           20    60.0          1.0000               1.0000             28.0 

/var/folders/_1/p0t70_y97x96fvjdnly6tc8h0000gn/T/ipykernel_2507/1220050512.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Resource Summary

In [7]:
# Measure inference latency
dummy_input = torch.randn(1, 20, FEAT_DIM)
for _ in range(10):
    with torch.no_grad():
        model(dummy_input)

times = []
for _ in range(200):
    t0 = time.perf_counter()
    with torch.no_grad():
        model(dummy_input)
    times.append((time.perf_counter() - t0) * 1000)
avg_ms = np.mean(times)
std_ms = np.std(times)

param_count = sum(p.numel() for p in model.parameters())
model_size_kb = model_path.stat().st_size / 1024
pose_model_size_mb = (PROJECT_ROOT / "motion" / "models" / "pose_landmarker_lite.task").stat().st_size / (1024 * 1024)
data_size_kb = sum(f.stat().st_size for f in data_root.rglob("*.npz")) / 1024

resources = {
    "Model architecture": "TCN, 2 temporal blocks, 64 channels, kernel_size=5",
    "Parameters": f"{param_count:,}",
    "Model file size": f"{model_size_kb:.0f} KB",
    "Input dimensions": f"(20, 81) = 20 frames x (27 pos + 27 vel + 27 acc)",
    "Upper-body joints": "9 (nose, shoulders, elbows, wrists, hips)",
    "Output classes": "5 (idle, lpunch, rpunch, forward, backward)",
    "Training data": f"8 npz recordings, {data_size_kb:.0f} KB, {len(ds)} windows",
    "Training time": "~18 s (Apple Silicon CPU)",
    "TCN inference": f"{avg_ms:.2f} +/- {std_ms:.2f} ms/window",
    "Pose estimation model": f"MediaPipe Pose Landmarker Lite, {pose_model_size_mb:.1f} MB",
    "Hardware": "MacBook (Apple Silicon), Python 3.12, PyTorch 2.x, no GPU",
    "Dependencies": "pygame, opencv, mediapipe, torch, numpy, scikit-learn",
}

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.axis("off")
table = ax.table(
    cellText=[[k, v] for k, v in resources.items()],
    colLabels=["Resource", "Value"],
    cellLoc="left",
    loc="center",
    colWidths=[0.35, 0.65],
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.4)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor("#4C72B0")
        cell.set_text_props(color="white", fontweight="bold")
    elif row % 2 == 0:
        cell.set_facecolor("#f0f0f0")

ax.set_title("Resource Summary", fontsize=13, pad=20)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "resource_summary.png")
plt.show()
print(f"Saved: {FIGURES_DIR / 'resource_summary.png'}")

Saved: /Users/ivanj/Desktop/Gesture game/Gestra_final/results/figures/resource_summary.png


/var/folders/_1/p0t70_y97x96fvjdnly6tc8h0000gn/T/ipykernel_2507/4146225288.py:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. System Pipeline Diagram

In [8]:
fig, ax = plt.subplots(figsize=(14, 3.5))
ax.set_xlim(0, 14)
ax.set_ylim(0, 3.5)
ax.axis("off")

boxes = [
    (0.3, 1.2, "Webcam\n(30 fps)"),
    (2.2, 1.2, "MediaPipe\nPose"),
    (4.1, 1.2, "9 Upper-Body\nJoints"),
    (6.0, 1.2, "Normalize\n(shoulder center)"),
    (8.0, 1.2, "Pos + Vel + Acc\n(20 x 81)"),
    (10.1, 1.2, "TCN\n(93K params)"),
    (12.1, 1.2, "5-Class\nAction"),
]

box_w, box_h = 1.6, 1.2
colors_pipe = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0", "#E91E63", "#00BCD4", "#FF5722"]

for i, (x, y, text) in enumerate(boxes):
    rect = mpatches.FancyBboxPatch((x, y), box_w, box_h, boxstyle="round,pad=0.1",
                                     facecolor=colors_pipe[i % len(colors_pipe)], alpha=0.8,
                                     edgecolor="white", linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x + box_w/2, y + box_h/2, text, ha="center", va="center",
            fontsize=9, fontweight="bold", color="white")
    if i > 0:
        prev_x = boxes[i-1][0]
        ax.annotate("", xy=(x, y + box_h/2), xytext=(prev_x + box_w, y + box_h/2),
                    arrowprops=dict(arrowstyle="->", color="#333", lw=2))

game_boxes = [
    (10.1, -0.2, "Named\nAction"),
    (12.1, -0.2, "Pygame\nCharacter"),
]
for i, (x, y, text) in enumerate(game_boxes):
    rect = mpatches.FancyBboxPatch((x, y), box_w, box_h, boxstyle="round,pad=0.1",
                                     facecolor="#607D8B", alpha=0.8,
                                     edgecolor="white", linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x + box_w/2, y + box_h/2, text, ha="center", va="center",
            fontsize=9, fontweight="bold", color="white")

ax.annotate("", xy=(10.1 + box_w/2, -0.2), xytext=(12.1 + box_w/2, 1.2),
            arrowprops=dict(arrowstyle="->", color="#333", lw=2))
ax.annotate("", xy=(12.1, -0.2 + box_h/2), xytext=(10.1 + box_w, -0.2 + box_h/2),
            arrowprops=dict(arrowstyle="->", color="#333", lw=2))

ax.set_title("Gestra System Pipeline", fontsize=14, fontweight="bold", pad=15)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "system_pipeline.png")
plt.show()
print(f"Saved: {FIGURES_DIR / 'system_pipeline.png'}")

Saved: /Users/ivanj/Desktop/Gesture game/Gestra_final/results/figures/system_pipeline.png


/var/folders/_1/p0t70_y97x96fvjdnly6tc8h0000gn/T/ipykernel_2507/765728176.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
